In [ ]:
from pathlib import Path
import os,sys,json,subprocess,time,hashlib
OWN=Path.cwd()/"llm4rec_s2_rq2"
DATA=Path.cwd()/json.loads((OWN/"data-path.json").read_text())["relative"]
env=dict(os.environ,PYTHONHASHSEED="0")
records=[]
for relative in ["baseline_seed0","checks/baseline_seed0_repeat"]:
    dest=OWN/relative
    assert not dest.exists()
    cmd=[sys.executable,str(OWN/"inputs/starter.py"),"run","--data-dir",str(DATA),"--output-dir",str(dest)]
    t=time.perf_counter();r=subprocess.run(cmd,env=env,capture_output=True,text=True)
    (OWN/(relative.replace("/","_")+"-console.log")).write_text(r.stdout+r.stderr)
    assert r.returncode==0,r.stderr
    records.append({"command":cmd,"PYTHONHASHSEED":"0","seconds_whole_starter_run_including_all_four_methods_and_io":time.perf_counter()-t,"returncode":r.returncode})
for name in ["contexts.jsonl","protocol.json","predictions.csv","per_user.csv","metrics.json"]:
    assert (OWN/"baseline_seed0"/name).read_bytes()==(OWN/"checks/baseline_seed0_repeat"/name).read_bytes(),name
original=[json.loads(l) for l in (OWN/"baseline/contexts.jsonl").read_text().splitlines()]
seeded=[json.loads(l) for l in (OWN/"baseline_seed0/contexts.jsonl").read_text().splitlines()]
changed=[a["user_id"] for a,b in zip(original,seeded) if [i["id"] for i in a["candidate_assets"]]!=[i["id"] for i in b["candidate_assets"]]]
manifest={"runs":records,"seeded_starter_exact_repeat_parity":True,"candidate_lists_changed_from_unseeded_initial_run":changed,"initial_run_preserved":"baseline","selected_fixed_context_source":"baseline_seed0","protocol_hash":json.loads((OWN/"baseline_seed0/protocol.json").read_text())["protocol_hash"],"reason":"Fix Python hash seed across unchanged starter subprocesses to make floating-point set-iteration tie behavior reproducible. No starter edits."}
(OWN/"seeded-baseline-manifest.json").write_text(json.dumps(manifest,indent=2)+"\n")
print(json.dumps(manifest,indent=2))
